# Калибровка вероятностей: проверяем руками

Ноутбук к теме [`lesson.md`](lesson.md). Здесь ECE и диаграмма надёжности пишутся в
десять строк, численно проверяется, что логлосс и Брайер минимальны ровно на истинных
вероятностях (а accuracy — нет), температурная шкала и Платт подбираются вручную, и
отдельно считается то, ради чего всё это нужно: деньги.

In [1]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

C_BG, C_INK, C_GRAY = "#faf9f5", "#141413", "#b0aea5"
C_PANEL, C_ORANGE, C_BLUE, C_GREEN = "#e8e6dc", "#d97757", "#6a9bcc", "#788c5d"
plt.rcParams.update({
    "figure.facecolor": C_BG, "axes.facecolor": C_BG, "axes.edgecolor": C_GRAY,
    "text.color": C_INK, "xtick.color": C_INK, "ytick.color": C_INK,
    "axes.labelcolor": C_INK, "font.size": 11, "legend.frameon": False,
})

def ece(y, p, bins=15, strategy="uniform"):
    '''Средневзвешенный по бинам разрыв |фактическая доля − средняя уверенность|.'''
    if strategy == "uniform":
        edges = np.linspace(0, 1, bins + 1)
    else:
        edges = np.quantile(p, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = 0, 1
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    return float(sum((idx == b).mean() * abs(y[idx == b].mean() - p[idx == b].mean())
                     for b in range(bins) if (idx == b).sum()))

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

## 1. Диаграмма надёжности своими руками

Обучаем три модели и смотрим не на точность, а на честность их чисел.

In [2]:
X, y = make_classification(n_samples=40_000, n_features=20, n_informative=8,
                           n_redundant=4, class_sep=0.8, flip_y=0.03, random_state=0)
Xtr, Xr, ytr, yr = train_test_split(X, y, test_size=0.7, random_state=0, stratify=y)
Xval, Xte, yval, yte = train_test_split(Xr, yr, test_size=0.5, random_state=0, stratify=yr)

models = {
    "логрегрессия": LogisticRegression(max_iter=2000).fit(Xtr, ytr),
    "случайный лес": RandomForestClassifier(n_estimators=300, random_state=0).fit(Xtr, ytr),
    "нейросеть": MLPClassifier((256, 256), max_iter=800, random_state=0).fit(Xtr, ytr),
}

print(f"{'модель':<16}{'accuracy':>10}{'AUC':>8}{'Brier':>9}{'логлосс':>10}{'ECE':>8}")
preds = {}
for name, m in models.items():
    p = m.predict_proba(Xte)[:, 1]
    preds[name] = p
    print(f"{name:<16}{m.score(Xte, yte):>10.3f}{roc_auc_score(yte, p):>8.3f}"
          f"{brier_score_loss(yte, p):>9.4f}{log_loss(yte, p):>10.4f}{ece(yte, p):>8.4f}")

print("\nдиаграмма надёжности для нейросети (бины равной массы по 10%):")
p = preds["нейросеть"]
edges = np.quantile(p, np.linspace(0, 1, 11))
print(f"{'бин':>6}{'средняя p':>12}{'фактическая доля':>20}{'разрыв':>10}{'объектов':>10}")
for b in range(10):
    m_ = (p >= edges[b]) & (p <= edges[b + 1] if b == 9 else p < edges[b + 1])
    print(f"{b + 1:>6}{p[m_].mean():>12.3f}{yte[m_].mean():>20.3f}"
          f"{yte[m_].mean() - p[m_].mean():>+10.3f}{m_.sum():>10}")

модель            accuracy     AUC    Brier   логлосс     ECE
логрегрессия         0.893   0.949   0.0825    0.2877  0.0187
случайный лес        0.948   0.979   0.0480    0.1919  0.0616
нейросеть            0.951   0.979   0.0422    0.3181  0.0349

диаграмма надёжности для нейросети (бины равной массы по 10%):
   бин   средняя p    фактическая доля    разрыв  объектов
     1       0.000               0.016    +0.016      1400
     2       0.000               0.009    +0.009      1400
     3       0.000               0.018    +0.018      1400
     4       0.000               0.034    +0.033      1400
     5       0.080               0.164    +0.083      1400
     6       0.940               0.834    -0.106      1400
     7       1.000               0.975    -0.025      1400
     8       1.000               0.974    -0.026      1400
     9       1.000               0.987    -0.013      1400
    10       1.000               0.984    -0.016      1400


Разрывы систематически отрицательные в верхних бинах: модель обещает больше, чем даёт.

## 2. Почему логлосс и Брайер «поощряют честность»

Строго правильная функция потерь (proper scoring rule) достигает минимума ожидаемого
значения ровно на истинной вероятности. Проверим численно: пусть истинная вероятность
события равна $p$, а мы заявляем $q$.

In [3]:
qs = np.linspace(0.01, 0.99, 99)
print(f"{'истинное p':>12}{'argmin логлосса':>18}{'argmin Брайера':>17}"
      f"{'argmin 0-1 (accuracy)':>24}")
for p_true in (0.2, 0.35, 0.5, 0.7, 0.9):
    ll = -(p_true * np.log(qs) + (1 - p_true) * np.log(1 - qs))
    br = p_true * (1 - qs) ** 2 + (1 - p_true) * qs ** 2
    err = p_true * (qs <= 0.5) + (1 - p_true) * (qs > 0.5)     # ожидаемая доля ошибок
    best_err = qs[err == err.min()]
    print(f"{p_true:>12.2f}{qs[np.argmin(ll)]:>18.2f}{qs[np.argmin(br)]:>17.2f}"
          f"{f'любое из [{best_err.min():.2f}, {best_err.max():.2f}]':>24}")

  истинное p   argmin логлосса   argmin Брайера   argmin 0-1 (accuracy)
        0.20              0.20             0.20   любое из [0.01, 0.50]
        0.35              0.35             0.35   любое из [0.01, 0.50]
        0.50              0.50             0.50   любое из [0.01, 0.99]
        0.70              0.70             0.70   любое из [0.51, 0.99]
        0.90              0.90             0.90   любое из [0.51, 0.99]


Логлосс и Брайер указывают точно на истинную вероятность; доле ошибок безразлично любое
значение по нужную сторону от 0.5 — поэтому accuracy и нельзя использовать как критерий
качества вероятностей.

## 3. Разложение Брайера и бесполезная калиброванная модель

In [4]:
def brier_decomposition(y, p, bins=15):
    '''BS = reliability − resolution + uncertainty.'''
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    base = y.mean()
    rel = res = 0.0
    for b in range(bins):
        m_ = idx == b
        if m_.sum():
            w = m_.mean()
            rel += w * (p[m_].mean() - y[m_].mean()) ** 2
            res += w * (y[m_].mean() - base) ** 2
    return rel, res, base * (1 - base)

p_const = np.full(len(yte), ytr.mean())
rows = [("константа p = доля класса", p_const)] + list(preds.items())
print(f"{'модель':<26}{'ECE':>8}{'AUC':>8}{'Brier':>9}{'rel':>9}{'res':>9}{'unc':>9}"
      f"{'сумма':>9}")
for name, p_ in rows:
    rel, res, unc = brier_decomposition(yte, p_)
    print(f"{name:<26}{ece(yte, p_):>8.4f}{roc_auc_score(yte, p_):>8.3f}"
          f"{brier_score_loss(yte, p_):>9.4f}{rel:>9.4f}{res:>9.4f}{unc:>9.4f}"
          f"{rel - res + unc:>9.4f}")

модель                         ECE     AUC    Brier      rel      res      unc    сумма
константа p = доля класса   0.0001   0.500   0.2500   0.0000   0.0000   0.2500   0.2500
логрегрессия                0.0187   0.949   0.0825   0.0006   0.1679   0.2500   0.0827
случайный лес               0.0616   0.979   0.0480   0.0062   0.2084   0.2500   0.0478
нейросеть                   0.0349   0.979   0.0422   0.0018   0.2091   0.2500   0.0427


Константа калибрована лучше всех и бесполезна: resolution ровно ноль, AUC 0.5.

## 4. Температурная шкала и Платт своими руками

Оба метода подбираются **на валидации**, которую модель не видела.

In [5]:
m = models["нейросеть"]
pv, pt = m.predict_proba(Xval)[:, 1], m.predict_proba(Xte)[:, 1]

def fit_temperature(y, p):
    z = logit(p)
    def nll(T):
        q = 1 / (1 + np.exp(-z / T))
        return -np.mean(y * np.log(q + 1e-12) + (1 - y) * np.log(1 - q + 1e-12))
    return float(minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x)

T = fit_temperature(yval, pv)
p_temp = 1 / (1 + np.exp(-logit(pt) / T))
platt = LogisticRegression().fit(logit(pv).reshape(-1, 1), yval)
p_platt = platt.predict_proba(logit(pt).reshape(-1, 1))[:, 1]

print(f"подобранная температура T = {T:.3f}  (T > 1 — модель переуверена)")
print(f"коэффициенты Платта: a = {platt.coef_[0][0]:.3f}, b = {platt.intercept_[0]:.3f}"
      f"   (a ≈ 1/T = {1 / T:.3f})\n")
print(f"{'вариант':<22}{'ECE':>9}{'Brier':>9}{'логлосс':>10}{'AUC':>10}")
for name, pp in (("без калибровки", pt), (f"температура T={T:.2f}", p_temp),
                 ("Платт", p_platt)):
    print(f"{name:<22}{ece(yte, pp):>9.4f}{brier_score_loss(yte, pp):>9.4f}"
          f"{log_loss(yte, pp):>10.4f}{roc_auc_score(yte, pp):>10.6f}")
d = roc_auc_score(yte, p_temp) - roc_auc_score(yte, pt)
print(f"\nразница AUC до и после калибровки: {d:+.2e}")
print("теоретически монотонное преобразование не меняет порядок и AUC вовсе;")
print("остаток в пятом знаке — от обрезки насыщенных вероятностей в logit(),")
print(f"их у модели немало: ровно 0 или 1 выдано для {np.mean((pt <= 1e-6) | (pt >= 1 - 1e-6)):.1%} объектов")

подобранная температура T = 2.768  (T > 1 — модель переуверена)
коэффициенты Платта: a = 0.361, b = -0.054   (a ≈ 1/T = 0.361)

вариант                     ECE    Brier   логлосс       AUC
без калибровки           0.0349   0.0422    0.3181  0.979386
температура T=2.77       0.0117   0.0402    0.1594  0.979428
Платт                    0.0114   0.0402    0.1592  0.979428

разница AUC до и после калибровки: +4.19e-05
теоретически монотонное преобразование не меняет порядок и AUC вовсе;
остаток в пятом знаке — от обрезки насыщенных вероятностей в logit(),
их у модели немало: ровно 0 или 1 выдано для 40.3% объектов


## 5. Что ломает калибровку: андерсэмплинг и обратная поправка

Если оставить долю $r$ отрицательных, шансы сдвигаются в $1/r$ раз. Вероятности надо
вернуть на место: $\text{odds}_{\text{true}} = r \cdot \text{odds}_{\text{model}}$.

In [6]:
rng = np.random.default_rng(0)
Xi, yi = make_classification(n_samples=60_000, n_features=15, n_informative=6,
                             weights=[0.97, 0.03], flip_y=0.01, random_state=1)
Xa, Xb, ya, yb = train_test_split(Xi, yi, test_size=0.4, random_state=0, stratify=yi)

full = LogisticRegression(max_iter=2000).fit(Xa, ya)
r = 0.05
neg = np.where(ya == 0)[0]
keep = np.concatenate([np.where(ya == 1)[0], rng.choice(neg, int(r * len(neg)), replace=False)])
under = LogisticRegression(max_iter=2000).fit(Xa[keep], ya[keep])

p_full = full.predict_proba(Xb)[:, 1]
p_und = under.predict_proba(Xb)[:, 1]
odds = p_und / (1 - p_und)
p_fix = (r * odds) / (1 + r * odds)

print(f"фактическая доля положительных в тесте: {yb.mean():.4f}\n")
print(f"{'модель':<28}{'средний скор':>14}{'ECE':>9}{'Brier':>9}{'AUC':>10}")
for name, pp in (("на всех данных", p_full), (f"undersampling r={r}", p_und),
                 ("он же после поправки", p_fix)):
    print(f"{name:<28}{pp.mean():>14.4f}{ece(yb, pp):>9.4f}"
          f"{brier_score_loss(yb, pp):>9.4f}{roc_auc_score(yb, pp):>10.4f}")

фактическая доля положительных в тесте: 0.0348

модель                        средний скор      ECE    Brier       AUC
на всех данных                      0.0357   0.0061   0.0192    0.8658
undersampling r=0.05                0.2419   0.2072   0.0908    0.8805
он же после поправки                0.0333   0.0060   0.0240    0.8805


## 6. Зачем всё это: деньги

Правило «блокировать, если $\hat p \cdot \text{сумма} > C$». Искажаем вероятности
монотонно — ранжирование не меняется вообще — и смотрим на потери.

In [7]:
rng = np.random.default_rng(3)
n = 60_000
p_true = rng.beta(1.2, 30, n)
y_fr = (rng.random(n) < p_true).astype(int)
amount = np.exp(rng.normal(3.2, 1.1, n))
C = 4.0

def cost(block):
    return float(C * np.sum(block & (y_fr == 0)) + np.sum(amount[(~block) & (y_fr == 1)]))

base = cost(np.zeros(n, bool))
print(f"доля события {y_fr.mean():.4f}, потери при бездействии {base:.0f}\n")
print(f"{'T':>6}{'ECE':>9}{'AUC':>9}{'алертов':>10}{'потери':>10}{'% базы':>9}")
for T in (0.4, 0.7, 1.0, 1.5, 3.0):
    p_hat = 1 / (1 + np.exp(-logit(p_true) / T))
    block = p_hat * amount > C
    c = cost(block)
    print(f"{T:>6}{ece(y_fr, p_hat):>9.4f}{roc_auc_score(y_fr, p_hat):>9.4f}"
          f"{block.mean():>10.2%}{c:>10.0f}{c / base * 100:>8.0f}%")

доля события 0.0380, потери при бездействии 100035

     T      ECE      AUC   алертов    потери   % базы
   0.4   0.0369   0.7380     0.07%     97477      97%
   0.7   0.0254   0.7380     2.30%     80058      80%
   1.0   0.0008   0.7380    10.21%     69084      69%
   1.5   0.0570   0.7380    28.86%     86473      86%
   3.0   0.1956   0.7380    60.93%    146288     146%


## 7. Что осталось в руках

- Калибровка — это совпадение заявленной уверенности с фактической частотой; ECE и
  диаграмма надёжности пишутся в десять строк.
- Логлосс и Брайер минимальны ровно на истинных вероятностях, а доле ошибок безразлично
  любое значение по нужную сторону от 0.5 — поэтому accuracy не критерий для вероятностей.
- Константа, равная базовой частоте, идеально калибрована и бесполезна: resolution ноль.
- Температурная шкала — один параметр, подбирается на валидации, снижает ECE в разы и
  практически не трогает AUC (остаток в пятом знаке — от обрезки насыщенных вероятностей);
  коэффициент Платта при логите оказывается почти равен $1/T$.
- Андерсэмплинг сдвигает шансы ровно в $1/r$ раз, и обратная поправка возвращает
  вероятности, не трогая ранжирование.
- При одном и том же AUC решение по некалиброванным вероятностям может стоить дороже,
  чем вообще ничего не делать.